## Cartpole Skating

> **Problem**: Agar Peter ko wolf se bachna hai, toh use wolf se tez chalne mein capable hona hoga. Hum dekhenge ki Peter skate karna kaise seekh sakta hai, khaaskar balance banaye rakhna, Q-Learning use karke.

Sabse pehle, gym install karein aur zaroori libraries import karein:

In [ ]:
import sys
!pip install gym pygame

import gym
import matplotlib.pyplot as plt
import numpy as np
import random

## Ek CartPole environment banayein

In [ ]:
env = gym.make("CartPole-v1")
print(env.action_space)
print(env.observation_space)
print(env.action_space.sample())

Environment kaise kaam karta hai yeh dekhne ke liye, chalo 100 steps ki ek chhoti simulation run karte hain.

In [ ]:
env.reset()

for i in range(100):
   env.render()
   env.step(env.action_space.sample())
env.close()

Simulation ke dauraan, humein yeh decide karne ke liye observations chahiye hote hain ki kaise act karna hai. Asal mein, `step` function humein current observation, reward function, aur `done` flag wapas deta hai jo batata hai ki simulation jaari rakhna samajhdari hai ya nahi:

In [ ]:
env.reset()

done = False
while not done:
   env.render()
   obs, rew, done, info = env.step(env.action_space.sample())
   print(f"{obs} -> {rew}")
env.close()

Hum un numbers ki minimum aur maximum value nikaal sakte hain:

In [ ]:
print(env.observation_space.low)
print(env.observation_space.high)

In [ ]:
def discretize(x):
    return tuple((x/np.array([0.25, 0.25, 0.01, 0.1])).astype(np.int))

Chalo bins use karke dusre discretization method ko bhi explore karte hain:

In [ ]:
def create_bins(i,num):
    return np.arange(num+1)*(i[1]-i[0])/num+i[0]

print("Sample bins for interval (-5,5) with 10 bins\n",create_bins((-5,5),10))

ints = [(-5,5),(-2,2),(-0.5,0.5),(-2,2)] # intervals of values for each parameter
nbins = [20,20,10,10] # number of bins for each parameter
bins = [create_bins(ints[i],nbins[i]) for i in range(4)]

def discretize_bins(x):
    return tuple(np.digitize(x[i],bins[i]) for i in range(4))

Ab chalo ek chhoti simulation run karte hain aur un discrete environment values ko observe karte hain.

In [ ]:
env.reset()

done = False
while not done:
   #env.render()
   obs, rew, done, info = env.step(env.action_space.sample())
   #print(discretize_bins(obs))
   print(discretize(obs))
env.close()

In [ ]:
Q = {}
actions = (0,1)

def qvalues(state):
    return [Q.get((state,a),0) for a in actions]

In [ ]:
# hyperparameters
alpha = 0.3
gamma = 0.9
epsilon = 0.90

In [ ]:
def probs(v,eps=1e-4):
    v = v-v.min()+eps
    v = v/v.sum()
    return v

Qmax = 0
cum_rewards = []
rewards = []
for epoch in range(100000):
    obs = env.reset()
    done = False
    cum_reward=0
    # == do the simulation ==
    while not done:
        s = discretize(obs)
        if random.random()<epsilon:
            # exploitation - chose the action according to Q-Table probabilities
            v = probs(np.array(qvalues(s)))
            a = random.choices(actions,weights=v)[0]
        else:
            # exploration - randomly chose the action
            a = np.random.randint(env.action_space.n)

        obs, rew, done, info = env.step(a)
        cum_reward+=rew
        ns = discretize(obs)
        Q[(s,a)] = (1 - alpha) * Q.get((s,a),0) + alpha * (rew + gamma * max(qvalues(ns)))
    cum_rewards.append(cum_reward)
    rewards.append(cum_reward)
    # == Periodically print results and calculate average reward ==
    if epoch%5000==0:
        print(f"{epoch}: {np.average(cum_rewards)}, alpha={alpha}, epsilon={epsilon}")
        if np.average(cum_rewards) > Qmax:
            Qmax = np.average(cum_rewards)
            Qbest = Q
        cum_rewards=[]

In [ ]:
plt.plot(rewards)

Is graph se kuch bhi batana mumkin nahi hai, kyunki stochastic training process ki nature ki wajah se training sessions ki length mein bahut variation hota hai. Is graph ko zyada samajhne layak banane ke liye, hum experiments ki series par **running average** calculate kar sakte hain, maan lijiye 100. Isse `np.convolve` use karke aasaani se kiya ja sakta hai:

In [ ]:
def running_average(x,window):
    return np.convolve(x,np.ones(window)/window,mode='valid')

plt.plot(running_average(rewards,100))

## Hyperparameters mein badlav karna aur result dekhna

Ab yeh dekhna dilchasp hoga ki trained model asal mein kaise behave karta hai. Chalo simulation run karte hain, aur hum wahi action selection strategy follow karenge jo training ke dauraan apnayi thi: Q-Table mein probability distribution ke hisaab se sampling karna:

In [ ]:
obs = env.reset()
done = False
while not done:
   s = discretize(obs)
   env.render()
   v = probs(np.array(qvalues(s)))
   a = random.choices(actions,weights=v)[0]
   obs,_,done,_ = env.step(a)
env.close()

## Result ko animated GIF mein save karna

Agar aap apne doston ko impress karna chahte hain, toh aap unhe balancing pole ki animated GIF picture bhej sakte hain. Aisa karne ke liye, hum `env.render` se ek image frame bana sakte hain, aur phir un frames ko PIL library se animated GIF mein save kar sakte hain:

In [ ]:
from PIL import Image
obs = env.reset()
done = False
i=0
ims = []
while not done:
   s = discretize(obs)
   img=env.render(mode='rgb_array')
   ims.append(Image.fromarray(img))
   v = probs(np.array([Qbest.get((s,a),0) for a in actions]))
   a = random.choices(actions,weights=v)[0]
   obs,_,done,_ = env.step(a)
   i+=1
env.close()
ims[0].save('images/cartpole-balance.gif',save_all=True,append_images=ims[1::2],loop=0,duration=5)
print(i)

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->